# TiA 5 — Pretraining, transfer and model scale

**Big question:** What has a pretrained model already learned, and how reusable is it?

The default experiment performs genuine pretraining and frozen-feature transfer on the packaged scikit-learn digits data. It is deliberately small and offline. The extension links to standard pretrained vision weights; it is not required to run this notebook.

## How to work through this activity

This is a guided investigation rather than a coding tutorial. For each experiment:

1. Read the mathematical claim and identify the quantity being measured.
2. Predict the qualitative result before running the code.
3. Run one cell at a time and inspect both values and plots.
4. Change only the suggested variable; rerun and explain what changed.
5. Answer the **Explain** questions in your own words.

The code contains more comments than production software intentionally. You are not expected to memorise framework syntax. Focus on the relationship between assumptions, measurements and conclusions.

## Notation and prediction

Write a pretrained network as $f(x)=h_\psi(g_\theta(x))$, with encoder $g_\theta$ and source-task head $h_\psi$. A frozen linear probe learns only

$$\hat y=\arg\max_k\left(Wg_\theta(x)+b\right)_k,$$

so probe performance measures how linearly accessible downstream information already is in the representation. Pretraining examples and downstream test examples are separated below. Predict how random, narrow pretrained and wider pretrained encoders will behave when labels are scarce.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from comp0090 import seed_everything

# Fix every random-number generator so that your plots match the reference run.
# After completing the guided activity, change the seed to test robustness.
rng = seed_everything(7)

# The default path is designed for a CPU. Set this to False only after the
# notebook works and you want to run longer variants.
FAST_MODE = True
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

In [ ]:
X,y=load_digits(return_X_y=True); X=(X/16).astype("float32"); xt=torch.tensor(X)
pretrain_idx,transfer_idx=train_test_split(np.arange(len(y)),test_size=.5,stratify=y,random_state=7)
class Encoder(nn.Module):
    def __init__(self,width): super().__init__(); self.features=nn.Sequential(nn.Linear(64,width),nn.ReLU(),nn.Linear(width,width),nn.ReLU()); self.head=nn.Linear(width,10)
    def forward(self,z,features=False):
        h=self.features(z); return h if features else self.head(h)
def pretrain(width,epochs=45):
    model=Encoder(width); opt=torch.optim.Adam(model.parameters(),lr=.01); target=torch.tensor(y[pretrain_idx])
    for _ in range(epochs): opt.zero_grad(); loss=nn.functional.cross_entropy(model(xt[pretrain_idx]),target); loss.backward();opt.step()
    return model
models={w:pretrain(w,25 if FAST_MODE else 60) for w in [8,32,96]}
random_model=Encoder(32)
features={"pixels":X,"random":random_model(xt,True).detach().numpy(),**{f"pretrained-{w}":m(xt,True).detach().numpy() for w,m in models.items()}}

## Frozen linear probes and label efficiency

The encoder is frozen. Only multinomial logistic regression sees downstream labels, isolating the information already present in each representation.

In [ ]:
# Only examples held out from pretraining enter this downstream split.
train,test=train_test_split(transfer_idx,test_size=.5,stratify=y[transfer_idx],random_state=7)
scores_by_count={}
for count in [5,20,60]:
    chosen=np.concatenate([train[y[train]==c][:count] for c in range(10)])
    scores={name:make_pipeline(StandardScaler(),LogisticRegression(max_iter=600)).fit(z[chosen],y[chosen]).score(z[test],y[test]) for name,z in features.items()}
    scores_by_count[count]=scores
    print(f"labels/class={count}",scores)
print("parameter counts",{w:sum(p.numel() for p in m.parameters()) for w,m in models.items()})
assert scores_by_count[5]["pretrained-96"] > scores_by_count[5]["random"]+.15
assert scores_by_count[20]["pretrained-96"] > scores_by_count[5]["pretrained-96"]

### Modern-model extension (optional)

Repeat the same `extract → freeze → probe` protocol with [torchvision's pretrained ResNet-18 and larger models](https://pytorch.org/vision/stable/models.html), or a [Vision Transformer](https://pytorch.org/vision/stable/models/vision_transformer.html). Those weights require an explicit download and substantially more memory, so they are not part of the default path. Record parameters, extraction time, memory, and probe accuracy—larger is a hypothesis, not a conclusion.

### Explain

1. Why can frozen features reduce labelled-data requirements?
2. What confound prevents this toy experiment from establishing a universal scaling law?
3. Distinguish pretraining, linear probing, and fine-tuning.

## Expected pattern and limits

Frozen pretrained features—especially from the wider encoder—should beat random features and become useful with few downstream labels. Increasing width also increases parameters; three tiny models cannot establish a scaling law. The optional modern-model comparison should control preprocessing and probing protocol.